# translation_function

Fonctions de construction des tables de traduction. Ce notebook ne produit
**aucune** table : il ne définit que des fonctions, appelées par
`build_translations` et `dim_language`.

Même rôle et même emplacement que `delta_function` : un notebook de fonctions,
chargé par `%run`, partagé entre les notebooks d'alimentation.

In [0]:
def ensure_delta_table(df, table_name):
    """Crée la table Delta cible si elle n'existe pas encore (schéma seul, 0 ligne).

    handle_table_update ne sait pas créer une table : DeltaTable.forName échoue si
    elle est absente. Ce bootstrap rend les notebooks rejouables sur un
    environnement neuf (dev, preprd) sans création manuelle préalable.

    La table est créée avec le schéma du DataFrame plus created_at, colonne que
    handle_table_update ajoute systématiquement. Pas d'updated_at : en mode "full"
    la fonction fait delete() puis write.mode("append"), et un append Delta exige
    des schémas qui correspondent exactement.
    """
    if not spark.catalog.tableExists(table_name):
        (
            df.limit(0)
              .withColumn("created_at", F.current_timestamp())
              .write.format("delta")
              .saveAsTable(table_name)
        )
        print(f"Table {table_name} créée (schéma seul, 0 ligne).")

## `build_translation_dim`

Rend dense une table `*_translations` source. Voir `build_translations` pour le
détail du raisonnement ; en résumé, trois problèmes sont traités ici :

1. le **fallback** du PO (langue → anglais → clé) n'est pas réalisable en RLS,
   qui sait supprimer des lignes mais pas en substituer une ;
2. les traductions sont **incomplètes** : une clé non traduite disparaîtrait du
   visuel après filtrage RLS, et les totaux seraient faux ;
3. le couple (clé, langue) doit être **unique** pour qu'il reste exactement une
   ligne par clé après filtrage.

In [0]:
# Langue de repli (English = 2 dans parameters_languages)
LANG_EN = 2


def build_translation_dim(df_trad, dim_language, key_cols, label_col="label", fallback_expr=None):
    """Rend dense une table *_translations source.

    Retourne une ligne par (clé x langue de dim_language), avec :
      - label        : libellé garanti non nul (langue -> anglais -> clé)
      - label_source : d'où vient le libellé, pour le contrôle qualité

    dim_language est passé en paramètre plutôt que lu dans le contexte global :
    c'est lui qui définit le périmètre des langues, et le rendre explicite évite
    une dépendance cachée à l'ordre des %run.
    """
    actives = df_trad.filter(F.col("deleted") == False)

    # 1. Dédoublonnage : une seule ligne par (clé, langue), la plus récente.
    #    Sans cela un doublon en base dupliquerait les lignes dans tous les visuels.
    w = Window.partitionBy(*key_cols, "language").orderBy(
        F.coalesce(F.col("updated_at"), F.col("created_at")).desc_nulls_last()
    )
    actives = (
        actives
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    # 2. Grille dense : toutes les clés x toutes les langues du périmètre.
    #    Le produit cartésien porte sur dim_language, pas sur les langues présentes
    #    dans la table source : c'est ce qui garantit qu'une langue non traduite
    #    obtient malgré tout une ligne (avec repli) au lieu de disparaître.
    keys = actives.select(*key_cols).distinct()
    grid = keys.crossJoin(F.broadcast(dim_language.select("language")))

    # 3. Libellé anglais, support du repli de niveau 2
    label_en = (
        actives.filter(F.col("language") == LANG_EN)
               .select(*key_cols, F.col(label_col).alias("_label_en"))
    )

    # 4. Repli de niveau 3 : la clé elle-même, jamais de null
    if fallback_expr is None:
        fallback_expr = F.concat_ws("_", *[F.col(c).cast("string") for c in key_cols])

    return (
        grid
        .join(
            actives.select(*key_cols, "language", F.col(label_col).alias("_label")),
            key_cols + ["language"], "left"
        )
        .join(label_en, key_cols, "left")
        .withColumn("label", F.coalesce(F.col("_label"), F.col("_label_en"), fallback_expr))
        .withColumn(
            "label_source",
            F.when(F.col("_label").isNotNull(), F.lit("translated"))
             .when(F.col("_label_en").isNotNull(), F.lit("fallback_en"))
             .otherwise(F.lit("fallback_key"))
        )
        .select(*key_cols, "language", "label", "label_source")
    )

## `publish_translation_dim`

Écriture avec la mécanique Delta du projet, en `mode="full"` imposé.

En mode `update`, `handle_table_update` ne fait qu'insérer et mettre à jour : il
ne supprime jamais. Une clé retirée à la source resterait donc indéfiniment dans
la table du modèle et continuerait d'apparaître dans les slicers du rapport — un
libellé fantôme, sans données derrière. Ces tables sont petites et entièrement
redérivées à chaque run : le remplacement complet est plus sûr et sans coût
notable.

In [0]:
# Noms des tables effectivement écrites, alimenté par publish_translation_dim.
# Sert au contrôle de couverture, qui relit les tables au lieu de recalculer les
# DataFrames.
tables_publiees = []


def publish_translation_dim(df, process_name, primary_key, target_catalog_schema):
    """Écrit une table de traduction et mémorise sa cible.

    target_catalog_schema est passé en paramètre : ces tables vivent dans le
    catalogue commun, pas dans le schéma du projet appelant.
    """
    target = f"{target_catalog_schema}.{process_name}"
    all_columns = df.columns
    additional_columns = get_additional_columns(all_columns, primary_key)

    if verbose_mode == 'debug':
        print(f"{target} -> clé {primary_key}, colonnes {all_columns}")

    ensure_delta_table(df, target)

    handle_table_update(
        df,
        target,
        primary_key,
        all_columns,
        additional_columns_to_check=additional_columns,
        mode="full"
    )

    tables_publiees.append((process_name, target))
    return target